In [33]:
import pandas as pd
import numpy as np
import yfinance as yf
from  datetime import date, timedelta
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense,LSTM

In [ ]:
today=date.today()
d1=today.strftime("%Y-%m-%d")
end_dtae=d1
d2 = date.today() - timedelta(days=5000)
d2=d2.strftime("%Y-%m-%d")
start_date=d2
# AAPL is Apple company 
data = yf.download(
    "AAPL",
    start=start_date,
    end=end_dtae,
    progress=False
)

# Flatten MultiIndex columns
if hasattr(data.columns, "nlevels") and data.columns.nlevels > 1:
    data.columns = data.columns.get_level_values(0)

print(data.head())
print(data.columns)



Price           Close       High        Low       Open     Volume
Date                                                             
2012-10-08  19.185913  19.468214  19.123981  19.447770  637994000
2012-10-09  19.116159  19.255656  18.746373  19.200339  838597200
2012-10-10  19.268290  19.390650  19.150740  19.233114  510356000
2012-10-11  18.883171  19.457394  18.883171  19.436349  546081200
2012-10-12  18.931572  19.102034  18.798991  18.927062  460014800
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


In [35]:
data["Date"] = data.index
data = data[["Date", "Open", "High", "Low", "Close", "Volume"]]
data.reset_index(drop=True, inplace=True)

In [36]:
print(data.tail())

Price       Date        Open        High         Low       Close    Volume
3436  2026-06-10  290.739990  294.750000  287.380005  291.579987  52793300
3437  2026-06-11  293.720001  297.000000  289.589996  295.630005  42572500
3438  2026-06-12  296.029999  297.140015  289.619995  291.130005  38742100
3439  2026-06-15  294.119995  297.779999  291.700012  296.420013  45732600
3440  2026-06-16  295.250000  300.480011  293.970001  299.239990  39830400


In [37]:


figure = go.Figure(
    data=[
        go.Candlestick(
            x=data["Date"],
            open=data["Open"],
            high=data["High"],
            low=data["Low"],
            close=data["Close"]
        )
    ]
)

figure.update_layout(title="Apple Stock Prices",
    xaxis_rangeslider_visible=False
)

figure.show()

In [38]:
print(data[["Open","High","Low","Close"]].isnull().sum())

Price
Open     0
High     0
Low      0
Close    0
dtype: int64


In [39]:
print(data.columns)

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='object', name='Price')


In [40]:
correlation=data.corr()
print(correlation["Close"].sort_values(ascending=False))

Price
Close     1.000000
Low       0.999890
High      0.999887
Open      0.999754
Date      0.941356
Volume   -0.540767
Name: Close, dtype: float64


In [ ]:
x=data[["Open", "High", "Low","Volume"]]
 
y=data["Close"]
x=x.to_numpy()
y=y.to_numpy()
y=y.reshape(-1,1)
xtrain, xtest, ytrain, ytest = train_test_split(
    x, y,
    train_size=0.8,
    shuffle=False
)

In [42]:
model=Sequential()
model.add(LSTM(128,return_sequences=True,input_shape=(xtrain.shape[1],1)))
model.add(LSTM(64,return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))
model.summary()

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 4, 128)         │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 25)             │         1,625 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,619 (459.45 KB)

 Trainable params: 117,619 (459.45 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
model.compile(optimizer="adam",loss="mean_squared_error")
model.fit(xtrain,ytrain,batch_size=1,epochs=30)


Epoch 1/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 1059.4352
Epoch 2/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 44.0212
Epoch 3/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 59.5828
Epoch 4/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - loss: 31.7401
Epoch 5/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 29.0140
Epoch 6/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 31.2518
Epoch 7/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 25.6695
Epoch 8/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 27.9581
Epoch 9/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 27.3271
Epoch 10/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 25.3750
Epoch 11/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 27.0926
Epoch 12/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 28.5095
Epoch 13/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 24.9201
Epoch 14/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 27.8033
Epoch 1

In [51]:
# Features sequence= Open,High,low,Volume
features = np.array([[180.08, 184.99, 175.07, 74919600]])
prediction = model.predict(features)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step
[[173.35767]]
